# ESZA019 — Visão Computacional  
## Laboratório 6 — Mapa de Profundidade com Câmera Estéreo

**Grupo 2**  
**Autores:** Cesar de Jesus, Mariana Chiara e Vinicius de Marchi  
**Docente:** Prof. Celso Setsuo Kurashima  
**Data de realização dos experimentos:** 20 de agosto de 2026  
**Data de publicação:** 20 de agosto de 2026  

---

### Resumo

Neste laboratório foi construída e calibrada uma câmera estéreo de baixo custo formada por duas webcams fixadas rigidamente. Foram capturados 25 pares de imagens de um tabuleiro de calibração com 9 × 6 cantos internos e quadrados de 3 cm. Após a calibração individual e estéreo, as imagens foram retificadas e processadas com o algoritmo StereoSGBM do OpenCV. A disparidade foi convertida em coordenadas tridimensionais por meio da matriz de reprojeção \(Q\), permitindo medir a distância entre o sistema estéreo e um objeto específico.

A validação foi realizada nas distâncias reais de 60, 80, 100 e 120 cm, com cinco repetições por distância. O sistema apresentou erro absoluto médio de **0,845 cm**, erro percentual médio de **1,159%**, RMSE de **1,028 cm** e coeficiente \(R^2=0,9980\), demonstrando boa linearidade, repetibilidade e precisão na faixa analisada.

## Sumário

1. Introdução  
2. Objetivos  
3. Fundamentação teórica  
4. Materiais e métodos  
5. Implementação computacional em Jupyter  
6. Resultados  
7. Análise e discussão  
8. Conclusões  
9. Referências

## 1. Introdução

A visão estereoscópica estima a estrutura tridimensional de uma cena a partir de duas imagens adquiridas por câmeras posicionadas em locais diferentes. De maneira semelhante à visão binocular humana, um ponto do espaço aparece em posições horizontais distintas nas imagens esquerda e direita. A diferença entre essas posições é denominada **disparidade**.

Para que a disparidade possa ser calculada de maneira consistente, é necessário determinar os parâmetros intrínsecos de cada câmera, a posição relativa entre elas e as transformações de retificação. Depois da retificação, pontos correspondentes passam a ser buscados ao longo da mesma linha horizontal. A disparidade obtida pode então ser convertida em profundidade, desde que a distância focal e a separação entre as câmeras sejam conhecidas.

Este relatório apresenta todo o procedimento empregado pelo Grupo 2: montagem do sistema, captura das imagens de calibração, calibração individual e estéreo, retificação, ajuste do algoritmo StereoSGBM, geração dos mapas de disparidade e profundidade, medição de distâncias, reconstrução 3D e validação quantitativa.

## 2. Objetivos

O objetivo geral foi desenvolver e validar um sistema estéreo capaz de estimar a distância entre as câmeras e objetos presentes na cena.

Objetivos específicos:

- compreender os conceitos de geometria epipolar, disparidade, retificação e profundidade;
- calibrar individualmente as duas webcams;
- estimar a rotação e a translação entre as câmeras;
- retificar os pares de imagens;
- gerar mapas densos de disparidade com Block Matching e StereoSGBM;
- converter disparidade em profundidade e coordenadas 3D;
- ajustar os parâmetros de correspondência para reduzir ruídos;
- medir distâncias conhecidas e calcular os erros experimentalmente;
- produzir uma nuvem de pontos da cena;
- documentar o procedimento de forma reprodutível.

## 3. Fundamentação teórica

### 3.1 Modelo de câmera e calibração

No modelo pinhole, um ponto tridimensional \((X,Y,Z)\) é projetado no plano da imagem por uma matriz de câmera composta por parâmetros intrínsecos e extrínsecos. A matriz intrínseca de cada câmera é escrita como

\[
K=
\begin{bmatrix}
f_x & 0 & c_x\\
0 & f_y & c_y\\
0 & 0 & 1
\end{bmatrix},
\]

em que \(f_x\) e \(f_y\) são as distâncias focais em pixels e \((c_x,c_y)\) é o ponto principal. A calibração também estima coeficientes de distorção radial e tangencial.

A calibração estéreo acrescenta a rotação \(R\) e a translação \(T\) entre os referenciais das duas câmeras. A norma de \(T\) corresponde à **baseline** \(B\), isto é, à distância entre os centros ópticos.

### 3.2 Geometria epipolar

Para um ponto tridimensional observado por duas câmeras, as projeções nas imagens obedecem à restrição epipolar

\[
\mathbf{x}_R^{T}F\mathbf{x}_L=0,
\]

em que \(F\) é a matriz fundamental. Essa restrição reduz a busca por correspondências de uma região bidimensional para uma linha epipolar.

### 3.3 Retificação

A retificação aplica transformações projetivas às imagens para tornar as linhas epipolares horizontais e alinhadas. Após esse processo, um ponto na imagem esquerda deve encontrar seu correspondente na mesma coordenada vertical da imagem direita. Essa condição reduz o custo computacional e melhora a robustez do cálculo de disparidade.

### 3.4 Disparidade

Para imagens retificadas, a disparidade horizontal de um ponto é

\[
d=x_L-x_R.
\]

Objetos próximos produzem disparidades maiores, enquanto objetos distantes produzem disparidades menores. O mapa de disparidade é denso quando uma estimativa é calculada para a maior parte dos pixels.

### 3.5 Profundidade

A profundidade \(Z\) pode ser obtida por triangulação:

\[
Z=\frac{fB}{d},
\]

em que \(f\) é a distância focal da câmera retificada, \(B\) é a baseline e \(d\) é a disparidade. Portanto, profundidade e disparidade possuem relação inversa.

No experimento, a matriz \(Q\), gerada por `cv2.stereoRectify`, foi empregada em `cv2.reprojectImageTo3D` para recuperar \((X,Y,Z)\) diretamente para cada pixel válido.

### 3.6 Diagrama de blocos do procedimento

![Diagrama de blocos](assets/diagrama_blocos.png)

### 3.7 Gráfico Profundidade × Disparidade

Para a calibração final, a baseline foi \(B=5,507\) cm e a distância focal retificada foi aproximadamente \(f=778,265\) pixels. Assim, \(fBpprox4285,61\ 	ext{px·cm}\).

![Profundidade versus disparidade](assets/grafico_profundidade_disparidade.png)

O formato hiperbólico mostra por que pequenas variações de disparidade podem produzir grandes variações de profundidade em objetos mais distantes.

## 4. Materiais e métodos

### 4.1 Materiais

- duas webcams USB;
- suporte rígido para impedir alteração da posição relativa;
- computador com Linux;
- tabuleiro de calibração com 9 × 6 cantos internos;
- quadrados físicos de 3 cm;
- régua ou trena para validação das distâncias;
- Python 3;
- OpenCV;
- NumPy;
- Matplotlib;
- Jupyter Notebook.

A resolução utilizada durante a calibração e a reconstrução foi de 640 × 480 pixels.

### 4.2 Captura dos pares de calibração

Foram adquiridos 25 pares válidos de imagens. O tabuleiro foi posicionado em diferentes regiões do campo de visão, distâncias e inclinações. Essa diversidade é importante porque uma sequência formada por imagens quase idênticas não fornece informação geométrica suficiente para estimar os parâmetros de maneira robusta.

Os cantos foram detectados simultaneamente nas duas imagens com `cv2.findChessboardCorners` e refinados em nível subpixel com `cv2.cornerSubPix`.

![Exemplo de detecção dos cantos](assets/calibracao_cantos_detectados.png)

### 4.3 Calibração individual e estéreo

A calibração foi executada em três etapas:

1. calibração da câmera esquerda com `cv2.calibrateCamera`;
2. calibração da câmera direita com `cv2.calibrateCamera`;
3. calibração estéreo com `cv2.stereoCalibrate`, mantendo os parâmetros intrínsecos fixos por meio de `cv2.CALIB_FIX_INTRINSIC`.

A matriz de rotação \(R\), o vetor de translação \(T\), as matrizes essencial \(E\) e fundamental \(F\) foram calculados. Em seguida, `cv2.stereoRectify` gerou \(R_1\), \(R_2\), \(P_1\), \(P_2\) e \(Q\).

Os parâmetros foram salvos em `stereo_params_abc.xml`.

### 4.4 Verificação da retificação

A qualidade da retificação foi verificada sobrepondo linhas horizontais ao par retificado. Quinas e cantos correspondentes do tabuleiro devem atravessar aproximadamente a mesma linha nas duas imagens.

![Par retificado](assets/retificacao_linhas_epipolares.png)

### 4.5 Cálculo da disparidade

Inicialmente foram testados os pares Tsukuba e Motorcycle disponibilizados como exemplo. O Block Matching permitiu observar o princípio de correspondência, enquanto o StereoSGBM produziu mapas mais densos e suaves.

| Par de exemplo | Resultado |
|---|---|
| Tsukuba com StereoBM | ![Tsukuba](assets/exemplo_tsukuba_bm.png) |
| Motorcycle com StereoSGBM | ![Motorcycle](assets/exemplo_motorcycle_sgbm.png) |

Para os dados próprios foi utilizado `cv2.StereoSGBM_create` no modo `STEREO_SGBM_MODE_SGBM_3WAY`.

### 4.6 Parâmetros finais do StereoSGBM

| Parâmetro | Valor final | Influência principal |
|---|---:|---|
| `minDisparity` | 0 | início da faixa de busca |
| `numDisparities` | 96 | extensão horizontal da busca; múltiplo de 16 |
| `blockSize` | 11 | suavidade do mapa e preservação de detalhes |
| `uniquenessRatio` | 18 | rejeição de correspondências ambíguas |
| `speckleWindowSize` | 180 | remoção de pequenas regiões isoladas |
| `speckleRange` | 2 | tolerância de disparidade nos speckles |
| `disp12MaxDiff` | 1 | consistência esquerda-direita |
| `preFilterCap` | 31 | limitação dos valores pré-filtrados |

O `blockSize=11` foi escolhido para produzir profundidades mais uniformes em superfícies planas. O aumento desse parâmetro reduz ruídos, porém pode suavizar bordas e objetos pequenos.

Quando o módulo `ximgproc` está disponível, o código utiliza filtro WLS. Caso contrário, aplica filtro de mediana 5 × 5. A distância local é calculada pela mediana de uma janela de 25 × 25 pixels, diminuindo a sensibilidade a pixels isolados.

### 4.7 Procedimento de validação

O tabuleiro foi posicionado nas distâncias reais de 60, 80, 100 e 120 cm, medidas a partir da região central entre as lentes até o plano do objeto. Para cada distância foram registradas cinco leituras em regiões centrais do mapa.

Foram calculados:

\[
E_{	ext{abs}}=|\overline Z-Z_{	ext{real}}|
\]

e

\[
E_{\%}=rac{|\overline Z-Z_{	ext{real}}|}{Z_{	ext{real}}}	imes100\%.
\]

O desvio padrão das cinco repetições foi utilizado como indicador de repetibilidade.

### 4.8 Comandos principais utilizados

```bash
python3 calibracao_nova/codigos/01_capturar_pares_calibracao.py   --cam-esq 0 --cam-dir 2 --colunas 9 --linhas 6
```

```bash
python3 calibracao_nova/codigos/02_calibrar_estereo.py   --pasta calibracao_nova/capturas   --colunas 9 --linhas 6   --quadrado-cm 3.0   --saida calibracao_nova/saida/stereo_params_abc.xml
```

```bash
python3 codigos/06_mapa_profundidade_ao_vivo.py   --cam-esq 0 --cam-dir 2   --calibracao calibracao/stereo_params_abc.xml   --fator-unidade-cm 1.0   --filtro auto   --raio-medicao 12   --z-min-cm 20   --z-max-cm 200
```

```bash
python3 codigos/05_processar_par_salvo.py   --esquerda capturas/par_002_20260720_193525_113_L.png   --direita capturas/par_002_20260720_193525_113_R.png   --calibracao calibracao/stereo_params_abc.xml   --config config_sgbm.json   --fator-unidade-cm 1.0   --saida resultados/tabuleiro_60cm
```

## 5. Implementação computacional em Jupyter

A seção a seguir constitui um programa completo e reprodutível para:

1. carregar a calibração;
2. retificar um par estéreo;
3. calcular a disparidade;
4. reprojetar os pixels para 3D;
5. medir a distância do tabuleiro em uma região de interesse.

In [1]:
from pathlib import Path
import json
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path("codigos").resolve()))
from lab6_util import (
    carregar_calibracao_xml,
    carregar_params_json,
    retificar_par,
    calcular_disparidade,
    reprojetar_3d,
)

PASTA = Path(".")
calibracao = carregar_calibracao_xml(PASTA / "dados" / "stereo_params_abc.xml")
params = carregar_params_json(PASTA / "dados" / "config_sgbm.json")

print("Resolução calibrada:", calibracao.image_size)
print("Tamanho físico do quadrado:", calibracao.square_size, "cm")
print("Parâmetros SGBM:", params)

Resolução calibrada: (640, 480)
Tamanho físico do quadrado: 3.0 cm
Parâmetros SGBM: {'min_disparity': 0, 'num_disparities': 96, 'block_size': 11, 'uniqueness_ratio': 18, 'speckle_window_size': 180, 'speckle_range': 2, 'disp12_max_diff': 1, 'pre_filter_cap': 31}


In [2]:
imagem_esquerda = cv2.imread(str(PASTA / "assets" / "tabuleiro_esquerda_original.png"))
imagem_direita = cv2.imread(str(PASTA / "assets" / "tabuleiro_direita_original.png"))

rect_l, rect_r = retificar_par(imagem_esquerda, imagem_direita, calibracao)
gray_l = cv2.cvtColor(rect_l, cv2.COLOR_BGR2GRAY)
gray_r = cv2.cvtColor(rect_r, cv2.COLOR_BGR2GRAY)

disparidade, _ = calcular_disparidade(gray_l, gray_r, params, filtro="auto")
pontos3d, mascara_valida = reprojetar_3d(disparidade, calibracao.Q)

print("Pixels com profundidade válida:", int(np.count_nonzero(mascara_valida)))

Pixels com profundidade válida: 147223


In [3]:
# Região central ocupada pelo tabuleiro no par final.
x1, y1, x2, y2 = 200, 120, 600, 430

profundidade = np.abs(pontos3d[:, :, 2])
mascara_roi = mascara_valida[y1:y2, x1:x2] & np.isfinite(
    profundidade[y1:y2, x1:x2]
)
valores_roi = profundidade[y1:y2, x1:x2][mascara_roi]

distancia_roi = float(np.median(valores_roi))
p10, p90 = np.percentile(valores_roi, [10, 90])

print(f"Distância mediana na ROI: {distancia_roi:.2f} cm")
print(f"Faixa central P10–P90: {p10:.2f}–{p90:.2f} cm")

Distância mediana na ROI: 59.21 cm
Faixa central P10–P90: 57.14–61.28 cm


O valor calculado na região do tabuleiro ficou próximo de 60 cm, coerente com a distância nominal do ensaio. A mediana foi preferida à média porque regiões de oclusão, bordas e correspondências incorretas podem produzir valores extremos.

## 6. Resultados

### 6.1 Resultados da calibração

| Grandeza | Resultado |
|---|---:|
| Pares válidos | 25 |
| RMS da câmera esquerda | 0,1734 px |
| RMS da câmera direita | 0,1679 px |
| RMS estéreo | 1,5284 px |
| Baseline estimada | 5,507 cm |
| Distância focal retificada | 778,265 px |
| Resolução | 640 × 480 px |

Os baixos erros RMS individuais indicam boa localização dos cantos nas imagens. O RMS estéreo é maior porque inclui a consistência geométrica conjunta entre as duas câmeras, mas a validação de distância demonstra que o resultado foi adequado para a faixa analisada.

### 6.2 Resultado do par a aproximadamente 60 cm

![Painel final](assets/painel_resultados_tabuleiro.png)

O processamento do par final gerou 147.223 pixels 3D válidos. A profundidade global mediana foi 60,36 cm. Considerando somente uma região central associada ao tabuleiro, a mediana foi aproximadamente 59,21 cm.

### 6.3 Medições de distância

In [4]:
medicoes_df = pd.read_csv(PASTA / "dados" / "medicoes_distancia.csv")
colunas_exibir = [
    "Distância real (cm)",
    "Medição 1 (cm)",
    "Medição 2 (cm)",
    "Medição 3 (cm)",
    "Medição 4 (cm)",
    "Medição 5 (cm)",
    "Média (cm)",
    "Desvio padrão (cm)",
    "Erro absoluto (cm)",
    "Erro percentual (%)",
]
medicoes_df[colunas_exibir].round(3)

,Distância real (cm),Medição 1 (cm),Medição 2 (cm),Medição 3 (cm),Medição 4 (cm),Medição 5 (cm),Média (cm),Desvio padrão (cm),Erro absoluto (cm),Erro percentual (%)
0,60.0,57.9,58.3,59.0,58.4,59.4,58.60,0.596,1.40,2.333
1,80.0,82.3,81.9,80.9,81.3,80.9,81.46,0.623,1.46,1.825
2,100.0,99.5,99.5,99.8,100.1,99.8,99.74,0.251,0.26,0.260
3,120.0,119.8,119.7,120.0,118.9,120.3,119.74,0.522,0.26,0.217


### 6.4 Validação métrica

![Validação](assets/grafico_validacao.png)

O ajuste entre distância real \(x\) e distância medida \(y\) foi

\[
y=1{,}0085x-0{,}88,
\]

com

\[
R^2=0{,}9980.
\]

A inclinação próxima de 1 e o intercepto próximo de zero indicam boa correspondência entre o valor real e o estimado.

### 6.5 Erros

![Erros percentuais](assets/grafico_erro_percentual.png)

| Indicador global | Resultado |
|---|---:|
| Erro absoluto médio | 0,845 cm |
| Erro percentual médio | 1,159% |
| RMSE | 1,028 cm |
| \(R^2\) | 0,9980 |

O maior erro percentual ocorreu a 60 cm, com 2,333%. Nas distâncias de 100 e 120 cm, os erros percentuais foram inferiores a 0,3%.

### 6.6 Registro em vídeo

O vídeo abaixo documenta o par retificado, os parâmetros do StereoSGBM, o mapa de profundidade ao vivo e a medição sobre o tabuleiro.

<video controls width="760" poster="assets/video_capa.png">
  <source src="assets/Captura_tabuleiro.webm" type="video/webm">
  O navegador não suporta vídeo HTML5.
</video>

[Baixar ou abrir o vídeo do procedimento](assets/Captura_tabuleiro.webm)

### 6.7 Reconstrução tridimensional

O par final também foi convertido em uma nuvem de pontos no formato PLY. O arquivo contém as coordenadas \(X\), \(Y\) e \(Z\), além das cores RGB associadas aos pixels válidos.

[Arquivo da nuvem de pontos — tabuleiro a aproximadamente 60 cm](dados/nuvem_pontos_tabuleiro_60cm.ply)

## 7. Análise e discussão

A calibração individual apresentou erros inferiores a 0,18 pixel, resultado associado à boa detecção subpixel dos cantos do tabuleiro. A baseline estimada de 5,507 cm foi compatível com a separação física das webcams, confirmando que a escala métrica foi preservada porque os pontos do objeto foram construídos diretamente em centímetros.

Durante os testes iniciais, mapas muito fragmentados foram observados quando o intervalo de disparidades era excessivo, o bloco de correspondência era pequeno e a remoção de speckles estava desativada. O ajuste final resolveu grande parte desses problemas:

- `numDisparities=96` limitou a busca a uma faixa compatível com as distâncias do experimento;
- `blockSize=11` aumentou a uniformidade em superfícies planas;
- `uniquenessRatio=18` rejeitou correspondências ambíguas;
- `speckleWindowSize=180` removeu regiões pequenas e isoladas;
- `speckleRange=2` preservou superfícies com pouca variação de disparidade;
- a mediana espacial de 25 × 25 pixels reduziu oscilações pontuais.

As regiões pretas no mapa correspondem a pixels sem correspondência válida, oclusões, áreas com pouca textura ou regiões próximas das bordas. Esses pixels não devem ser interpretados como profundidade zero.

O erro não variou monotonicamente com a distância: a medição de 80 cm apresentou viés positivo de 1,46 cm, enquanto as distâncias de 100 e 120 cm apresentaram erros muito pequenos. Esse comportamento mostra que o erro depende não apenas da distância, mas também da textura local, iluminação, oclusões e quantização da disparidade.

A repetibilidade foi satisfatória. Os desvios padrão ficaram entre 0,251 e 0,623 cm. Portanto, diferentes cliques realizados sobre a mesma superfície produziram resultados próximos, especialmente quando evitadas bordas e regiões inválidas.

A relação \(Z=fB/d\) explica a sensibilidade do sistema: como a disparidade é quantizada em pixels, a incerteza de profundidade cresce quando a disparidade diminui. Em aplicações futuras, uma baseline maior, câmeras com maior resolução e sincronização de hardware podem ampliar a faixa útil e melhorar a precisão em objetos distantes.

## 8. Conclusões

Foi desenvolvido e validado um sistema de visão estéreo baseado em duas webcams e OpenCV. A calibração com 25 pares de imagens permitiu determinar os parâmetros intrínsecos, extrínsecos e a baseline de 5,507 cm. A retificação alinhou adequadamente as linhas epipolares e possibilitou o cálculo de mapas densos de disparidade.

Após a sintonia do StereoSGBM e a aplicação de filtragem espacial, o mapa de profundidade tornou-se suficientemente uniforme para medir superfícies planas. A validação entre 60 e 120 cm produziu erro absoluto médio de 0,845 cm, erro percentual médio de 1,159%, RMSE de 1,028 cm e \(R^2=0,9980\).

Os resultados demonstram que uma câmera estéreo de baixo custo pode fornecer medições quantitativas confiáveis em uma faixa limitada de distâncias, desde que:

- as câmeras permaneçam rigidamente fixadas;
- a calibração seja realizada após qualquer alteração de posição;
- as imagens sejam retificadas;
- os objetos apresentem textura suficiente;
- os parâmetros de correspondência sejam ajustados para a cena;
- as medições sejam feitas por estatística local, e não por um único pixel.

Além das distâncias, o sistema foi capaz de gerar uma nuvem de pontos colorida da cena, completando a reconstrução tridimensional proposta.

## 9.Referências

1. LEARNOPENCV. *Making A Low-Cost Stereo Camera Using OpenCV*. Disponível em: <https://learnopencv.com/making-a-low-cost-stereo-camera-using-opencv/>.

2. LEARNOPENCV. *Introduction to Epipolar Geometry and Stereo Vision*. Disponível em: <https://learnopencv.com/introduction-to-epipolar-geometry-and-stereo-vision/>.

3. LEARNOPENCV. *Stereo Camera Depth Estimation With OpenCV*. Disponível em: <https://learnopencv.com/depth-perception-using-stereo-camera-python-c/>.

4. OPENCV. *Depth Map from Stereo Images*. Disponível em: <https://docs.opencv.org/4.x/dd/d53/tutorial_py_depthmap.html>.

5. LOOP, C.; ZHANG, Z. *Computing Rectifying Homographies for Stereo Vision*. IEEE Conference on Computer Vision and Pattern Recognition, 1999.

6. LEARNOPENCV. *Geometry of Image Formation*. Disponível em: <https://learnopencv.com/geometry-of-image-formation/>.

7. SZELISKI, R. *Computer Vision: Algorithms and Applications*. 2. ed. Springer, 2022.

---

### Arquivos anexos

- `dados/stereo_params_abc.xml`: calibração final;
- `dados/config_sgbm.json`: parâmetros finais de disparidade;
- `dados/medicoes_distancia.csv`: dados brutos e estatísticas;
- `dados/nuvem_pontos_tabuleiro_60cm.ply`: nuvem de pontos;
- `codigos/`: programas utilizados;
- `assets/Captura_tabuleiro.webm`: vídeo do procedimento.